# 00 — Shared Data Preparation

Cleans the raw review data once and writes a fixed, **stratified 85/15 train/validation split (seed = 42)** that all three models (TF-IDF, GloVe + RNN, BERT) must use.

**Do not create another split or re-clean in the model notebooks.** The cleaning here is deliberately light, so it is safe for every feature format (BERT keeps casing and punctuation; lowercasing, tokenizing, stopwords etc. are model-specific and happen in each model notebook).

## Steps
1. Load `data/raw/train.json` (labeled) and the test reviews from `data/raw/submission.csv` (there is no `test.json`)
2. Base cleaning: strip HTML tags, unescape HTML entities, normalize whitespace and unicode
3. Drop empty reviews, duplicate reviews and reviews with conflicting labels (**train only**; the test set keeps every row and its order, because `submission.csv` must match the template)
4. **Stratified** split so the positive/negative ratio is the same in train and validation
5. Verify (class ratios, no train/val overlap, no empties) and export to `data/processed/`

## Output files (`data/processed/`)
| File | Columns | Use |
|---|---|---|
| `train_split.json` | `reviews`, `sentiments` | model training |
| `val_split.json` | `reviews`, `sentiments` | model selection / evaluation |
| `test_split.json` | `reviews` (no labels) | final predictions, original row order |
| `prep_summary.json` | counts and ratios | for the report |


In [1]:
# ==================== SETUP & PATHS ====================
import html
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_RATIO = 0.85
np.random.seed(SEED)

RAW_DIR = Path("data/raw")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"seed: {SEED} | train ratio: {TRAIN_RATIO}")
print(f"Raw data: {RAW_DIR.resolve()}")
print(f"Output:   {OUT_DIR.resolve()}")

seed: 42 | train ratio: 0.85
Raw data: /Users/brymat24/repos/IE4483-AI-Data-Mining-Project/data/raw
Output:   /Users/brymat24/repos/IE4483-AI-Data-Mining-Project/data/processed


In [2]:
# ==================== LOAD RAW DATA ====================
with open(RAW_DIR / "train.json", "r", encoding="utf-8") as f:
    df_train_all = pd.DataFrame(json.load(f))

# There is no test.json: train.json is the only labeled data (split into train/validation below).
# The test set is the unlabeled reviews in the submission template, in template order.
df_test = pd.read_csv(RAW_DIR / "submission.csv")

# Keep the review text only; the test set must carry no predictions.
df_test = df_test[["reviews"]].copy()

assert {"reviews", "sentiments"} <= set(df_train_all.columns), "train.json needs reviews + sentiments"
assert df_train_all["sentiments"].isin([0, 1]).all(), "sentiments must be 0/1"

print(f"Raw train: {len(df_train_all)} | raw test: {len(df_test)}")
print(df_train_all["sentiments"].value_counts().rename({0: "Negative", 1: "Positive"}).to_string())

Raw train: 7401 | raw test: 1851
sentiments
Positive    6319
Negative    1082


In [3]:
# ==================== BASE CLEANING (shared by all models) ====================
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")

def clean_text(text) -> str:
    """Model-agnostic cleaning. Keeps casing and punctuation (BERT needs them);
    lowercasing / tokenization / stopwords are left to each model notebook."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)                  # &amp; -> &
    text = TAG_RE.sub(" ", text)                # strip HTML tags
    text = unicodedata.normalize("NFKC", text)  # unify unicode forms (curly quotes, nbsp, ...)
    text = WS_RE.sub(" ", text).strip()         # collapse spaces/newlines/tabs
    return text

df_train_all["reviews"] = df_train_all["reviews"].map(clean_text)
df_test["reviews"] = df_test["reviews"].map(clean_text)   # cleaned only: no rows dropped, order kept

n_raw = len(df_train_all)

# 1) empty reviews
df_clean = df_train_all[df_train_all["reviews"] != ""]
n_empty = n_raw - len(df_clean)

# 2) reviews that appear more than once with different labels are ambiguous: drop all copies
label_count = df_clean.groupby("reviews")["sentiments"].transform("nunique")
df_clean = df_clean[label_count == 1]
n_conflict = n_raw - n_empty - len(df_clean)

# 3) exact duplicates (after cleaning): keep the first copy
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="reviews", keep="first").reset_index(drop=True)
n_dup = before - len(df_clean)

print(f"Raw train rows:            {n_raw}")
print(f"  dropped empty:           {n_empty}")
print(f"  dropped conflicting:     {n_conflict}")
print(f"  dropped duplicates:      {n_dup}")
print(f"Clean train rows:          {len(df_clean)}")
print(f"Test rows (all kept):      {len(df_test)} | empty after cleaning: {(df_test['reviews'] == '').sum()}")

Raw train rows:            7401
  dropped empty:           0
  dropped conflicting:     0
  dropped duplicates:      219
Clean train rows:          7182
Test rows (all kept):      1851 | empty after cleaning: 0


In [4]:
# ==================== STRATIFIED TRAIN / VALIDATION SPLIT ====================
# Duplicates were removed BEFORE splitting, so no review can sit in both train and validation.
# stratify=... keeps the Negative/Positive ratio the same in both splits.
df_train, df_val = train_test_split(
    df_clean,
    train_size=TRAIN_RATIO,
    stratify=df_clean["sentiments"],
    random_state=SEED,
    shuffle=True,
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Training samples:   {len(df_train)}")
print(f"Validation samples: {len(df_val)}")
print(f"Test samples:       {len(df_test)}")

Training samples:   6104
Validation samples: 1078
Test samples:       1851


In [5]:
# ==================== VERIFICATION ====================
names = {0: "Negative", 1: "Positive"}
dist = pd.DataFrame({
    "Clean (%)": df_clean["sentiments"].value_counts(normalize=True) * 100,
    "Training (%)": df_train["sentiments"].value_counts(normalize=True) * 100,
    "Validation (%)": df_val["sentiments"].value_counts(normalize=True) * 100,
    "Training (n)": df_train["sentiments"].value_counts(),
    "Validation (n)": df_val["sentiments"].value_counts(),
}).rename(index=names).round(2)

print("=== Class distribution (Negative and Positive) ===")
display(dist)

# Both classes must match between train and validation (stratified: gap should be ~0).
for label in ("Negative", "Positive"):
    gap = abs(dist.loc[label, "Training (%)"] - dist.loc[label, "Validation (%)"])
    print(f"{label}: train vs validation gap = {gap:.2f} percentage points")
    assert gap < 0.1, f"{label} ratio differs too much between train and validation"

# No leakage and no bad rows.
assert set(df_train["reviews"]).isdisjoint(set(df_val["reviews"])), "train/val overlap"
assert df_train["reviews"].is_unique and df_val["reviews"].is_unique, "duplicates remain"
assert (df_train["reviews"] != "").all() and (df_val["reviews"] != "").all(), "empty review"
assert len(df_train) + len(df_val) == len(df_clean)
assert list(df_test.columns) == ["reviews"], "test must be unlabeled"
print("PASS: stratified, no overlap, no duplicates, no empty reviews.")

# Overlap with test is allowed (test rows must all be predicted) but worth knowing.
overlap = df_test["reviews"].isin(set(df_clean["reviews"])).sum()
print(f"Info: {overlap} test reviews also appear verbatim in the training data.")

# Length stats (words) to choose GloVe sequence length / BERT max_length in the model notebooks.
words = df_train["reviews"].str.split().str.len()
print("Words per review (train): " + ", ".join(
    f"p{q}={int(words.quantile(q / 100))}" for q in (50, 90, 95, 99)) + f", max={words.max()}")

=== Class distribution (Negative and Positive) ===


,Clean (%),Training (%),Validation (%),Training (n),Validation (n)
sentiments,,,,,
Positive,85.41,85.4,85.44,5213,921
Negative,14.59,14.6,14.56,891,157


Negative: train vs validation gap = 0.04 percentage points
Positive: train vs validation gap = 0.04 percentage points
PASS: stratified, no overlap, no duplicates, no empty reviews.
Info: 101 test reviews also appear verbatim in the training data.
Words per review (train): p50=41, p90=107, p95=141, p99=268, max=1066


In [6]:
# ==================== EXPORT ====================
summary = {
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "raw_train_rows": int(n_raw),
    "dropped_empty": int(n_empty),
    "dropped_conflicting": int(n_conflict),
    "dropped_duplicates": int(n_dup),
    "clean_rows": int(len(df_clean)),
    "train_rows": int(len(df_train)),
    "val_rows": int(len(df_val)),
    "test_rows": int(len(df_test)),
    "positive_ratio": {
        "clean": round(float(df_clean["sentiments"].mean()), 4),
        "train": round(float(df_train["sentiments"].mean()), 4),
        "val": round(float(df_val["sentiments"].mean()), 4),
    },
}

outputs = {
    "train_split.json": df_train,
    "val_split.json": df_val,
    "test_split.json": df_test,
}
for filename, frame in outputs.items():
    frame.to_json(OUT_DIR / filename, orient="records", force_ascii=False)

with open(OUT_DIR / "prep_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Re-load and check what was written.
for filename, frame in outputs.items():
    path = OUT_DIR / filename
    assert path.exists(), f"Missing file: {path}"
    assert len(pd.read_json(path, orient="records")) == len(frame), f"Row count mismatch: {filename}"
    print(f"Verified: {path} ({len(frame)} rows)")
print(json.dumps(summary, indent=2))

Verified: data/processed/train_split.json (6104 rows)
Verified: data/processed/val_split.json (1078 rows)
Verified: data/processed/test_split.json (1851 rows)
{
  "seed": 42,
  "train_ratio": 0.85,
  "raw_train_rows": 7401,
  "dropped_empty": 0,
  "dropped_conflicting": 0,
  "dropped_duplicates": 219,
  "clean_rows": 7182,
  "train_rows": 6104,
  "val_rows": 1078,
  "test_rows": 1851,
  "positive_ratio": {
    "clean": 0.8541,
    "train": 0.854,
    "val": 0.8544
  }
}


## Loading the shared data in a model notebook

```python
train = pd.read_json("data/processed/train_split.json")   
val   = pd.read_json("data/processed/val_split.json")
test  = pd.read_json("data/processed/test_split.json")    # reviews only, original order
```